Урок в прозе: https://proproprogs.ru/nn_pytorch/pytorch-klassy-sequential-modulelist-i-moduledict

Sequential определяет цепочку слоев. причем он может выступать самостоятельной моделью, так как метод forward там переопределен автоматически

In [1]:
import matplotlib.pyplot as plt

import torch
import torch.utils.data as data
import torchvision.transforms.v2 as tfs
import torchvision
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
from torchvision.datasets import ImageFolder

In [16]:
model = nn.Sequential(
    nn.Linear(28 * 28, 32),
    nn.ReLU(),
    nn.Linear(32, 10)
)

Но в этом случае к слоям нельзя будет обратиться по именам атрибутов, а иногда это требуется

In [17]:
model.__dict__

{'training': True,
 '_parameters': {},
 '_buffers': {},
 '_non_persistent_buffers_set': set(),
 '_backward_pre_hooks': OrderedDict(),
 '_backward_hooks': OrderedDict(),
 '_is_full_backward_hook': None,
 '_forward_hooks': OrderedDict(),
 '_forward_hooks_with_kwargs': OrderedDict(),
 '_forward_hooks_always_called': OrderedDict(),
 '_forward_pre_hooks': OrderedDict(),
 '_forward_pre_hooks_with_kwargs': OrderedDict(),
 '_state_dict_hooks': OrderedDict(),
 '_state_dict_pre_hooks': OrderedDict(),
 '_load_state_dict_pre_hooks': OrderedDict(),
 '_load_state_dict_post_hooks': OrderedDict(),
 '_modules': {'0': Linear(in_features=784, out_features=32, bias=True),
  '1': ReLU(),
  '2': Linear(in_features=32, out_features=10, bias=True)}}

In [10]:
# только так
getattr(model, '0')

Linear(in_features=784, out_features=32, bias=True)

In [11]:
model.layer_1

AttributeError: 'Sequential' object has no attribute 'layer_1'

In [12]:
model._modules

{'0': Linear(in_features=784, out_features=32, bias=True),
 '1': ReLU(),
 '2': Linear(in_features=32, out_features=10, bias=True)}

тогда можно объявить модель так

In [13]:
model = nn.Sequential()
model.add_module('layer_1', nn.Linear(28 * 28, 32))
model.add_module('relu', nn.ReLU())
model.add_module('layer_2', nn.Linear(32, 10))
model.__dict__

{'training': True,
 '_parameters': {},
 '_buffers': {},
 '_non_persistent_buffers_set': set(),
 '_backward_pre_hooks': OrderedDict(),
 '_backward_hooks': OrderedDict(),
 '_is_full_backward_hook': None,
 '_forward_hooks': OrderedDict(),
 '_forward_hooks_with_kwargs': OrderedDict(),
 '_forward_hooks_always_called': OrderedDict(),
 '_forward_pre_hooks': OrderedDict(),
 '_forward_pre_hooks_with_kwargs': OrderedDict(),
 '_state_dict_hooks': OrderedDict(),
 '_state_dict_pre_hooks': OrderedDict(),
 '_load_state_dict_pre_hooks': OrderedDict(),
 '_load_state_dict_post_hooks': OrderedDict(),
 '_modules': {'layer_1': Linear(in_features=784, out_features=32, bias=True),
  'relu': ReLU(),
  'layer_2': Linear(in_features=32, out_features=10, bias=True)}}

In [4]:
model.layer_1

Linear(in_features=784, out_features=32, bias=True)

Sequential может быть составной частью более сложной модели

In [14]:
class DigitNN(nn.Module):
    def __init__(self, input_dim, num_hidden, output_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, num_hidden),
            nn.ReLU(),
            nn.Linear(num_hidden, output_dim),
        )
 
    def forward(self, x):
        return self.net(x)

# ModuleList

если мы хотим задать слои через список, нам нужно сделать это через  nn.ModuleList, иначе слои не будут видны. Дело в том, что слои регистрируются только в том случае, если на них ведет какая-либо переменная. А если слой не зарегистрирован, его веса не будут обучаться

In [2]:
class ModelNN(nn.Module):
    def __init__(self, input_dim, output_dim, n_layers=3):
        super().__init__()
        self.layers = nn.ModuleList(
            [nn.Linear(input_dim // n, input_dim // (n+1)) for n in range(1, n_layers+1)]
        )
        sz_input = self.layers[-1].out_features
        self.layer_out = nn.Linear(sz_input, output_dim)
 
    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
            x = nn.functional.tanh(x)
 
        x = self.layer_out(x)
        return x

In [3]:
model = ModelNN(28*28, 10)
model

ModelNN(
  (layers): ModuleList(
    (0): Linear(in_features=784, out_features=392, bias=True)
    (1): Linear(in_features=392, out_features=261, bias=True)
    (2): Linear(in_features=261, out_features=196, bias=True)
  )
  (layer_out): Linear(in_features=196, out_features=10, bias=True)
)

# ModuleDict

In [ ]:
class ModelNN(nn.Module):
    def __init__(self, input_dim, output_dim, n_layers=3, act_type=None):
        super().__init__()
        self.act_type = act_type
        self.layers = nn.ModuleList()
 
        for n in range(1, n_layers+1):
            self.layers.add_module(f'layer_{n}', nn.Linear(input_dim // n, input_dim // (n+1)))
 
        self.layer_out = nn.Linear(input_dim // (n_layers+1), output_dim)
 
        self.act_lst = nn.ModuleDict({
            'relu': nn.ReLU(),
            'lk_relu': nn.LeakyReLU(),
        })
 
    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
            if self.act_type and self.act_type in self.act_lst:
                x = self.act_lst[self.act_type](x)
 
        x = self.layer_out(x)
        return x